<a href="https://colab.research.google.com/github/Mc-cloud/chessRL/blob/main/agents/Agent_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 55.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=81a943f4c8b90141fbf446ed66998f4db9f27460764c50c2b2ea210083668bb5
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [ ]:
import math
import random
import chess
import copy

class Node:
  def __init__(self, state : chess.Board, parent = None, prior_prob = 0.0):
    self.state = state
    self.parent = parent
    self.children = {}

    self.n_visits = 0
    self.value_sum = 0
    self.q_value = 0
    self.prior_prob = prior_prob

  def expand(self, action_probs):
    for move, prob in action_probs.items():
      if move not in self.children:
        next_state = self.state.copy()
        next_state.push(chess.Move.from_uci(move))

        self.children[move] = Node(state=next_state, parent= self, prior_prob = prob)

  def is_expended(self):
    return len(self.children) > 0

  def best_child(self,c):
    best_score = -math.inf
    best_action = None
    best_child = None

    for action, child in self.children.items():
      q_val = child.q_value
      u_val = c * child.prior_prob * math.sqrt(self.n_visits)/ (1 + child.n_visits)
      puct_score = q_val + u_val

      if puct_score > best_score :
        best_score = puct_score
        best_action = action
        best_child = child

    return best_action, best_child

  def backpropagate(self, value):
      self.n_visits += 1
      self.value_sum += value
      self.q_value = self.value_sum / self.n_visits

      if self.parent is not None:
        self.parent.backpropagate(-value)

In [ ]:
class MCTS:
  def __init__(self, neural_net, c = 1.5, n_simulations = 800):
    self.nn = neural_net
    self.c = c
    self.n_simulations = n_simulations

  def search(self, initial_state : chess.Board):
    root = Node(state = initial_state)

    for _ in range(self.n_simulations):
      node = root

      while node.is_expended():
        action, node = node.best_child(self.c)

      if node.state.is_game_over():
        value = -1.0 if node.state.is_checkmate() else 0.0
      else :
        action_probs, value = self.nn.predict(node.state)

        legal_moves = [m.uci() for m in node.state.legal_moves]
        legal_probs = {m : prob for m, prob in action_probs.items() if m in legal_moves}

        sum_probs = sum(legal_probs.values())

        if sum_probs > 0:
          legal_probs = {m: prob / sum_probs for m, prob in legal_probs.items()}
        else :
          legal_probs = {m : 1.0/len(legal_moves) for m in legal_moves}

        node.expand(legal_probs)

      node.backpropagate(-value)

    action_visits = {action : child.n_visits for action, child in root.children.items()}
    sum_visits = sum(action_visits.values())

    mcts_policy = {action : visits/sum_visits for action, visits in action_visits.items()}

    return mcts_policy

In [ ]:
import torch.nn as nn
import torch
import torch.nn.functional as F

class CNN(nn.Module):
  def __init__(self, input_channels, board_size, action_size, hidden_dim = 64):
    super().__init__()
    self.board_size = board_size
    self.action_size = action_size

    self.conv1 = nn.Conv2d(input_channels, hidden_dim, kernel_size=3, padding = 1)
    self.conv2 = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
    self.conv4 = nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1)

    self.bn1 = nn.BatchNorm2d(hidden_dim)
    self.bn2 = nn.BatchNorm2d(hidden_dim)
    self.bn3 = nn.BatchNorm2d(hidden_dim)
    self.bn4 = nn.BatchNorm2d(hidden_dim)

    self.policy_conv = nn.Conv2d(hidden_dim, 2, kernel_size=1)
    self.policy_bn = nn.BatchNorm2d(2)
    self.policy_fc = nn.Linear(2 * board_size * board_size, action_size)

    self.value_conv = nn.Conv2d(hidden_dim, 1, kernel_size=1)
    self.value_bn = nn.BatchNorm2d(1)
    self.value_fc1 = nn.Linear(1 * board_size * board_size, 64)
    self.value_fc2 = nn.Linear(64, 1)

  def forward(self, x):
    x = F.relu(self.bn1(self.conv1(x)))
    x = F.relu(self.bn2(self.conv2(x)))
    x = F.relu(self.bn3(self.conv3(x)))
    x = F.relu(self.bn4(self.conv4(x)))

    p = F.relu(self.policy_bn(self.policy_conv(x)))
    p = p.view(p.size(0), -1)
    p = self.policy_fc(p)
    policy_out = F.log_softmax(p, dim=1)

    v = F.relu(self.value_bn(self.value_conv(x)))
    v = v.view(v.size(0), -1)
    v = F.relu(self.value_fc1(v))
    v = self.value_fc2(v)
    value_out = torch.tanh(v)

    return policy_out, value_out

  def predict(self, board):
    board_tensor = board_to_tensor(board)
    device = next(self.parameters()).device
    board_tensor = board_tensor.to(device)

    self.eval()
    with torch.no_grad():
      board_tensor = board_tensor.unsqueeze(0)
      log_policy, value = self.forward(board_tensor)
      policy = torch.exp(log_policy).squeeze(0).cpu().numpy()
      value = value.item()

    policy_dict = {}
    for i in range(len(policy)):
        # On utilise le dictionnaire IDX_TO_UCI généré précédemment
        if i in IDX_TO_UCI:
            move_uci = IDX_TO_UCI[i]
            policy_dict[move_uci] = policy[i]

    return policy_dict, value



In [ ]:
import numpy as np
import torch
import chess

def board_to_tensor(board : chess.Board):
  tensor = np.zeros((13,8,8), dtype = np.float32)

  for square, piece in board.piece_map().items():
    rank = chess.square_rank(square)
    file = chess.square_file(square)

    channel = piece.piece_type - 1

    if piece.color == chess.BLACK:
      channel += 6

    tensor[channel, rank, file] = 1.0

  if board.turn == chess.WHITE:
    tensor[12, :, :] = 1.0
  else :
    tensor[12, :, :] = 0.0

  return torch.tensor(tensor)


In [ ]:
import chess

def create_move_vocab():
    """Crée les dictionnaires de traduction entre les coups UCI et les index du réseau."""
    uci_to_idx = {}
    idx_to_uci = {}
    idx = 0

    for from_sq in chess.SQUARES:
        for to_sq in chess.SQUARES:
            if from_sq == to_sq:
                continue

            from_rank = chess.square_rank(from_sq)
            to_rank = chess.square_rank(to_sq)

            is_promotion = (from_rank == 6 and to_rank == 7) or (from_rank == 1 and to_rank == 0)

            file_diff = abs(chess.square_file(from_sq) - chess.square_file(to_sq))

            if is_promotion and file_diff <= 1:
                for promo in ['q', 'r', 'b', 'n']:
                    move_uci = chess.SQUARE_NAMES[from_sq] + chess.SQUARE_NAMES[to_sq] + promo
                    uci_to_idx[move_uci] = idx
                    idx_to_uci[idx] = move_uci
                    idx += 1
            else:
                move_uci = chess.SQUARE_NAMES[from_sq] + chess.SQUARE_NAMES[to_sq]
                uci_to_idx[move_uci] = idx
                idx_to_uci[idx] = move_uci
                idx += 1

    return uci_to_idx, idx_to_uci

UCI_TO_IDX, IDX_TO_UCI = create_move_vocab()

def move_to_index(move : chess.Move):
  return UCI_TO_IDX[move.uci()]

def index_to_move(index : int, board : chess.Board):
  return chess.Move.from_uci(IDX_TO_UCI[index])

In [ ]:
import numpy as np
import torch
import chess

def play_one_game(neural_net, num_simulations = 800, temp_threshold  =15):
  board = chess.Board()
  memory = []
  move_count = 0

  mcts = MCTS(neural_net, n_simulations = num_simulations)
  while not board.is_game_over():
    move_count += 1
    policy_dict = mcts.search(board)
    action_size = len(UCI_TO_IDX)
    policy_vector = np.zeros(action_size, dtype = np.float32)

    for uci_move, prob in policy_dict.items():
      idx = UCI_TO_IDX[uci_move]
      policy_vector[idx] = prob

    memory.append((board_to_tensor(board), policy_vector, board.turn))
    actions = list(policy_dict.keys())
    probs = list(policy_dict.values())

    if move_count <= temp_threshold:
      chosen_uci = np.random.choice(actions, p=probs)
    else:
      chosen_uci = actions[np.argmax(probs)]

    board.push(chess.Move.from_uci(chosen_uci))

  result = board.result()
  if result == "1-0":
    winner = chess.WHITE
  elif result == "0-1":
    winner = chess.BLACK
  else :
    winner = None

  dataset = []
  for state_tensor, policy_vector, player_turn in memory:
    if winner is None :
      reward = 0.0  # Nul
    elif player_turn == winner:
      reward = 1.0  # Cette position a mené à une victoire pour ce joueur
    else:
      reward = -1.0 # Cette position a mené à une défaite

  dataset.append((state_tensor, policy_vector, reward))

  return dataset

def generate_games_with_self_play(neural_net, num_games=10, num_simulations=100, temp_threshold=15):
    """
    Joue plusieurs parties d'affilée contre soi-même et fusionne toutes
    les positions générées dans un seul grand dataset d'entraînement.
    """
    dataset_total = []

    print(f"\n🔄 Début de la génération de {num_games} parties en Self-Play...")

    for i in range(num_games):
        print(f"♟️ Partie {i+1}/{num_games} en cours...")

        game_data = play_one_game(
            neural_net,
            num_simulations=num_simulations,
            temp_threshold=temp_threshold
        )

        dataset_total.extend(game_data)

    print(f"✅ Génération terminée ! {len(dataset_total)} positions collectées.")

    return dataset_total

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class ChessDataset(Dataset):
    def __init__(self, dataset_total):
        self.dataset = dataset_total

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        state, policy, value = self.dataset[idx]
        return (
            state.clone().detach(),
            torch.tensor(policy, dtype=torch.float32),
            torch.tensor([value], dtype=torch.float32)
        )

def alpha_zero_loss(log_policy_preds, value_preds, policy_targets, value_targets):
  """
  Combine l'erreur sur le score (Value) et l'erreur sur les coups (Policy).
  """
  value_loss = F.mse_loss(value_preds, value_targets)

  policy_loss = -torch.sum(policy_targets * log_policy_preds, dim=1).mean()

  return value_loss + policy_loss

def train_network(neural_net, dataset_total, epochs=10, batch_size=64, learning_rate=0.001):
    """
    Prend le réseau actuel et l'entraîne sur les données du Self-Play.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    neural_net.to(device)

    optimizer = optim.Adam(neural_net.parameters(), lr=learning_rate, weight_decay=1e-4)

    dataset = ChessDataset(dataset_total)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    neural_net.train()

    print(f"Début de l'entraînement sur {device} avec {len(dataset_total)} positions...")

    for epoch in range(epochs):
        total_loss = 0.0

        for states, policies, values in dataloader:
            states = states.to(device)
            policies = policies.to(device)
            values = values.to(device)

            optimizer.zero_grad()

            log_policy_preds, value_preds = neural_net(states)

            loss = alpha_zero_loss(log_policy_preds, value_preds, policies, values)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss moyenne : {total_loss / len(dataloader):.4f}")

    neural_net.to("cpu")
    print("Entraînement terminé !")



In [ ]:
import chess

def evaluate_new_net(old_net, new_net, num_games=20, mcts_simulations=100, win_threshold=0.55):
    """
    Fait s'affronter l'ancien et le nouveau réseau.
    Retourne True si le nouveau réseau est significativement meilleur.
    """
    print(f"\n⚔️ Bienvenue dans l'Arène ! Début du match en {num_games} parties... ⚔️")
    new_wins = 0
    old_wins = 0
    draws = 0

    old_net.eval()
    new_net.eval()

    for i in range(num_games):
        board = chess.Board()

        # on alterne les couleurs.
        if i % 2 == 0:
            white_net = new_net
            black_net = old_net
            new_is_white = True
        else:
            white_net = old_net
            black_net = new_net
            new_is_white = False

        # On instancie des MCTS tout neufs pour vider leur mémoire entre chaque partie
        white_mcts = MCTS(white_net, n_simulations=mcts_simulations)
        black_mcts = MCTS(black_net, n_simulations=mcts_simulations)

        while not board.is_game_over():
            if board.turn == chess.WHITE:
                policy_dict = white_mcts.search(board)
            else:
                policy_dict = black_mcts.search(board)

            best_move = max(policy_dict, key=policy_dict.get)
            board.push(chess.Move.from_uci(best_move))

        result = board.result()
        if result == "1-0":
            if new_is_white: new_wins += 1
            else: old_wins += 1
        elif result == "0-1":
            if not new_is_white: new_wins += 1
            else: old_wins += 1
        else:
            draws += 1

        print(f"Partie {i+1}/{num_games} terminée | Score global -> Nouveau: {new_wins} | Ancien: {old_wins} | Nuls: {draws}")

    total_score = new_wins + (0.5 * draws)
    win_rate = total_score / num_games

    print(f"\n📊 Ratio de victoire du Challenger : {win_rate:.1%}")

    if win_rate >= win_threshold:
        print("👑 Succès ! Le Nouveau Réseau a surpassé le maître. Il devient le standard.")
        return True
    else:
        print("❌ Échec. Le Nouveau Réseau est rejeté. On garde l'Ancien pour la prochaine génération.")
        return False

In [ ]:
import copy

best_network = CNN(input_channels=13, board_size=8, action_size=len(UCI_TO_IDX))
iteration = 1
while True:
    print(f"=== GÉNÉRATION {iteration} ===")
    dataset = generate_games_with_self_play(best_network, num_games=1000)

    challenger_network = copy.deepcopy(best_network)

    train_network(challenger_network, dataset)
    is_better = evaluate_new_net(old_net=best_network, new_net=challenger_network)

    if is_better:
        best_network = challenger_network

    iteration += 1

=== GÉNÉRATION 1 ===

🔄 Début de la génération de 1000 parties en Self-Play...
♟️ Partie 1/1000 en cours...
♟️ Partie 2/1000 en cours...
♟️ Partie 3/1000 en cours...
